# 🏰 Capstone: Chat with Documents

**Module 6: Building RAG Systems** | **Type: Capstone (Castle)**

---

## Project Overview

Build a complete chat-with-documents application that processes a corpus, stores it in a vector database, and supports multi-turn conversations with source attribution and lost-in-the-middle mitigation.

## Learning Objectives

By the end of this capstone, you will have:

1. **Built** a document processing pipeline with structured loading and chunking
2. **Implemented** a vector store with ChromaDB for semantic search
3. **Created** a conversation memory system for multi-turn interactions
4. **Applied** lost-in-the-middle mitigation through smart context ordering
5. **Designed** a chat engine integrating retrieval, memory, and generation

## Concepts Applied

| Concept | From |
|---------|------|
| Context Retrieval | lab-basic-rag |
| Context Injection | lab-basic-rag |
| Context Stuffing | Module 6 concepts |
| Answer Generation | lab-basic-rag |
| Lost-in-the-Middle | Module 6 concepts |
| Source Citation | Module 6 concepts |

## Prerequisites

- **lab-basic-rag**: Complete RAG pipeline with embeddings, chunking, vector store, and retrieval
- **mini-embeddings**: Text embeddings and cosine similarity
- **mini-vector-db**: Vector database operations with ChromaDB

## What You'll Build

A multi-turn chat application that lets users ask questions about a document corpus, with conversation memory, context ordering for better answer quality, and a Gradio chat UI.

In [2]:
import json
from pathlib import Path
from typing import List, Dict, Optional
from dataclasses import dataclass, field
from openai import OpenAI
import chromadb
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
print('Setup complete!')

Setup complete!


## 2. Document Processing Pipeline 

In [4]:
@dataclass
class Document:
    '''A document with content and metadata.'''
    id: str
    content: str
    metadata: Dict = field(default_factory=dict)

class DocumentLoader:
    '''Load documents from various sources.'''
    
    def load_json(self, path: Path) -> List[Document]:
        '''Load documents from JSON file.'''
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        return [
            Document(
                id=d['id'],
                content=d['content'],
                metadata={'title': d.get('title', ''), 'type': d.get('type', '')}
            )
            for d in data
        ]

# Load documents
loader = DocumentLoader()
corpus_path = Path('../data/ai_engineering_docs')
all_docs = []

for file in corpus_path.glob('*.json'):
    docs = loader.load_json(file)
    all_docs.extend(docs)

print(f'Loaded {len(all_docs)} documents')

Loaded 70 documents


## 3. Text Chunker

In [5]:
@dataclass
class Chunk:
    '''A chunk of text with metadata.'''
    id: str
    text: str
    metadata: Dict

class TextChunker:
    '''Split documents into chunks.'''
    
    def __init__(self, chunk_size=400, overlap=50):
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def chunk_document(self, doc: Document) -> List[Chunk]:
        '''Split a document into chunks.'''
        text = doc.content
        chunks = []
        
        # Split by paragraphs first
        paragraphs = text.split('\n\n')
        current_chunk = ''
        chunk_idx = 0
        
        for para in paragraphs:
            if len(current_chunk) + len(para) < self.chunk_size:
                current_chunk += para + '\n\n'
            else:
                if current_chunk:
                    chunks.append(Chunk(
                        id=f'{doc.id}_chunk_{chunk_idx}',
                        text=current_chunk.strip(),
                        metadata={**doc.metadata, 'source_id': doc.id, 'chunk_idx': chunk_idx}
                    ))
                    chunk_idx += 1
                current_chunk = para + '\n\n'
        
        if current_chunk:
            chunks.append(Chunk(
                id=f'{doc.id}_chunk_{chunk_idx}',
                text=current_chunk.strip(),
                metadata={**doc.metadata, 'source_id': doc.id, 'chunk_idx': chunk_idx}
            ))
        
        return chunks

# Chunk all documents
chunker = TextChunker(chunk_size=400)
all_chunks = []
for doc in all_docs[:30]:  # Limit for demo
    chunks = chunker.chunk_document(doc)
    all_chunks.extend(chunks)

print(f'Created {len(all_chunks)} chunks from {len(all_docs[:30])} documents')

Created 123 chunks from 30 documents


## 4. Vector Store

In [6]:
class VectorStore:
    '''ChromaDB-based vector store.'''
    
    def __init__(self, collection_name='docs'):
        self.client = chromadb.Client()
        try:
            self.client.delete_collection(collection_name)
        except:
            pass
        self.collection = self.client.create_collection(collection_name)
    
    def add_chunks(self, chunks: List[Chunk]):
        '''Add chunks to the vector store.'''
        self.collection.add(
            ids=[c.id for c in chunks],
            documents=[c.text for c in chunks],
            metadatas=[c.metadata for c in chunks]
        )
    
    def search(self, query: str, n_results=5, filter_dict=None):
        '''Search for similar chunks.'''
        return self.collection.query(
            query_texts=[query],
            n_results=n_results,
            where=filter_dict
        )

# Create and populate vector store
vector_store = VectorStore('ai_engineering')
vector_store.add_chunks(all_chunks)
print(f'Added {len(all_chunks)} chunks to vector store')

Added 123 chunks to vector store


## 5. Conversation Memory

In [7]:
class ConversationMemory:
    '''Manage conversation history.'''
    
    def __init__(self, max_turns=10):
        self.history: List[Dict] = []
        self.max_turns = max_turns
    
    def add_turn(self, role: str, content: str):
        '''Add a conversation turn.'''
        self.history.append({'role': role, 'content': content})
        # Keep only recent turns
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-self.max_turns * 2:]
    
    def get_history(self) -> List[Dict]:
        '''Get conversation history for context.'''
        return self.history
    
    def get_summary(self) -> str:
        '''Get a summary of the conversation.'''
        if not self.history:
            return ''
        return '\n'.join([f"{h['role']}: {h['content'][:100]}..." for h in self.history[-4:]])
    
    def clear(self):
        '''Clear conversation history.'''
        self.history = []

memory = ConversationMemory()
print('Conversation memory initialized')

Conversation memory initialized


## 6. Chat Engine

In [8]:
class ChatEngine:
    '''Main chat engine for document Q&A.'''
    
    def __init__(self, vector_store: VectorStore, client: OpenAI):
        self.vector_store = vector_store
        self.client = client
        self.memory = ConversationMemory()
    
    def _build_context(self, results, max_chars=3000):
        '''Build context from search results.'''
        # Order: most relevant first and last (lost-in-the-middle mitigation)
        docs = results['documents'][0]
        metas = results['metadatas'][0]
        
        if len(docs) > 2:
            # Put best at start, second-best at end
            reordered = [docs[0]] + docs[2:] + [docs[1]]
            metas_reordered = [metas[0]] + metas[2:] + [metas[1]]
        else:
            reordered = docs
            metas_reordered = metas
        
        context_parts = []
        total = 0
        for doc, meta in zip(reordered, metas_reordered):
            if total + len(doc) > max_chars:
                break
            context_parts.append(f"[{meta.get('title', 'Source')}]\n{doc}")
            total += len(doc)
        
        return '\n\n---\n\n'.join(context_parts)
    
    def chat(self, user_message: str) -> Dict:
        '''Process a user message and generate response.'''
        
        # Search for relevant context
        results = self.vector_store.search(user_message, n_results=5)
        context = self._build_context(results)
        
        # Get conversation history
        history = self.memory.get_history()
        
        # Build system message
        system = '''You are a helpful AI assistant that answers questions based on the provided documentation.
Rules:
- Answer based on the context provided
- If you don't know, say so
- Be concise but thorough
- Cite sources when possible'''
        
        # Build user message with context
        user_prompt = f'''Context from documentation:
{context}

User question: {user_message}'''
        
        # Prepare messages
        messages = [{'role': 'system', 'content': system}]
        messages.extend(history)
        messages.append({'role': 'user', 'content': user_prompt})
        
        # Generate response
        response = self.client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            temperature=0.3
        )
        
        answer = response.choices[0].message.content
        
        # Update memory
        self.memory.add_turn('user', user_message)
        self.memory.add_turn('assistant', answer)
        
        return {
            'answer': answer,
            'sources': [m.get('title', '') for m in results['metadatas'][0][:3]]
        }

# Create chat engine
chat_engine = ChatEngine(vector_store, client)
print('Chat engine ready!')

Chat engine ready!


## 7. Interactive Chat

In [9]:
# Test the chat engine
questions = [
    'What is self-attention and how does it work?',
    'Can you explain more about the scaling factor?',  # Follow-up
    'How do I prevent prompt injection attacks?'
]

for q in questions:
    print(f'\n👤 User: {q}')
    result = chat_engine.chat(q)
    print(f'\n🤖 Assistant: {result["answer"]}')
    print(f'📚 Sources: {result["sources"]}')
    print('-' * 60)


👤 User: What is self-attention and how does it work?

🤖 Assistant: Self-attention, also known as intra-attention, is a mechanism that enables each position in a sequence to consider all other positions within the same sequence. This allows the model to compute representations by relating different positions, effectively capturing contextual information.

### How It Works:
1. **Input Sequence**: Given an input sequence \( X \), self-attention computes attention scores that determine the relevance of each position to every other position.
2. **Attention Mechanism**: It functions like a soft dictionary lookup:
   - **Query**: Represents what the model is looking for.
   - **Keys**: Represent the information contained in each position.
   - **Values**: Provide the actual information to be used based on the attention scores.
3. **Attention Weights**: These weights dictate how much each position's value contributes to the final output representation.

### Residual Connections:
In practice, 

## 8. Chat UI

We use **Gradio** to create a beautiful, modern chat interface with minimal code. It supports streaming and can be shared via public links.

### Gradio Chat Interface

In [10]:
# Gradio chat UI

import gradio as gr

def gradio_chat(message, history):
    """Wrapper for Gradio chat interface."""
    # Reset memory if new conversation
    if not history:
        chat_engine.memory.clear()
    
    result = chat_engine.chat(message)
    sources = ', '.join(result['sources'][:2]) if result['sources'] else 'No sources'
    return f"{result['answer']}\n\n📚 *Sources: {sources}*"

# Create and launch the interface
demo = gr.ChatInterface(
    fn=gradio_chat,
    title="📚 Chat with Documents",
    description="Ask questions about AI Engineering documentation",
    examples=[
        "What is self-attention?",
        "How do transformers work?",
        "Explain prompt injection attacks"
    ]
    )

# Launch inline in notebook
demo.launch(inline=True, height=500)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 🎯 Summary

### What You Built

You built a full chat-with-documents application that processes a corpus of AI engineering documentation, stores it in a vector database, and supports multi-turn conversations with source attribution. The system uses smart context ordering to mitigate the lost-in-the-middle problem, with a Gradio chat UI.

### Skills Practiced

| Skill | How It Was Applied |
|-------|-------------------|
| Document Processing | Loaded and structured documents with metadata using a DocumentLoader class |
| Text Chunking | Split documents into overlapping chunks using paragraph-aware splitting |
| Vector Database | Used ChromaDB to store and query document embeddings |
| Semantic Search | Retrieved relevant chunks based on query similarity |
| Context Management | Applied lost-in-the-middle reordering for better answer quality |
| Conversation Memory | Maintained multi-turn context with automatic history pruning |
| Answer Generation | Used retrieved context to generate grounded responses |
| Chat UI | Built a Gradio chat interface for interactive document Q&A |

### Key Takeaways

1. **End-to-end RAG** — chunking, embedding, retrieval, and generation form a complete pipeline that grounds LLM answers in real documents
2. **Context ordering matters** — placing the most relevant chunks at the start and end of the context improves answer quality
3. **Conversation memory** — maintaining history enables natural follow-up questions without repeating context